# Nuclear Data Fast MOOGP Models

This notebook fits four fast-mode MOOGP models on `moogp/nuclear_data` using the requested output blocks, an 80/20 train-test split, and either a shared fixed `q` or the fallback rule `q = ceil(25% * p_block)`.

Assumption: `all_f.csv` is the only available target matrix, so `RRMSE` is computed against `all_f` itself. Under that deterministic-output assumption, `RRMSE` will match `PRMSE`.

The default `maxiter` below is intentionally modest so the four-model run stays practical. Increase it if you want tighter optimization.

In [1]:
from pathlib import Path

import pandas as pd
import os
import sys

sys.path.append(os.path.abspath('..'))
from moogp.nuclear_data_experiment import load_nuclear_dataset, run_nuclear_fast_experiment

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


In [23]:
seed = 42
train_fraction = 0.8
fixed_q = 4  # e.g. 6
q_fraction = None  # used only when fixed_q is None
maxiter = 300
standardize_y = "zscore"
blocks = [(0, 30), (30, 54), (54, 78), (78, None)]


In [24]:
dataset = load_nuclear_dataset()

dataset_summary = pd.DataFrame(
    [
        {
            "data_dir": str(Path(dataset["data_dir"]).resolve()),
            "n_samples": dataset["X"].shape[0],
            "n_inputs": dataset["X"].shape[1],
            "n_outputs": dataset["Y"].shape[1],
        }
    ]
)

output_index_df = pd.DataFrame(
    [
        {
            "name": family.name,
            "start_col_1based": family.start + 1,
            "end_col_1based": family.end,
        }
        for family in dataset["output_index"]
    ]
)

display(dataset_summary)
display(output_index_df)


,data_dir,n_samples,n_inputs,n_outputs
0,/Users/evanbarnett/Desktop/Northwestern/Resear...,541,15,98


,name,start_col_1based,end_col_1based
0,dNch_deta,1,8
1,dET_deta,9,30
2,dN_dy_pion,31,38
3,dN_dy_kaon,39,46
4,dN_dy_proton,47,54
5,mean_pT_pion,55,62
6,mean_pT_kaon,63,70
7,mean_pT_proton,71,78
8,pT_fluct,79,90
9,v22,91,98


In [25]:
run = run_nuclear_fast_experiment(
    seed=seed,
    train_fraction=train_fraction,
    fixed_q=fixed_q,
    q_fraction=q_fraction,
    maxiter=maxiter,
    standardize_y=standardize_y,
    blocks=blocks,
)

model_summary_df = pd.DataFrame(run["model_summary"])
metrics_df = pd.DataFrame(run["metrics"])


In [26]:
model_summary_df[
    [
        "model",
        "column_range",
        "python_slice",
        "n_outputs",
        "q",
        "q_rule",
        "output_groups_label",
        "used_fast",
        "opt_success",
        "nit",
        "nfev",
        "nll",
    ]
].sort_values("model").reset_index(drop=True)


,model,column_range,python_slice,n_outputs,q,q_rule,output_groups_label,used_fast,opt_success,nit,nfev,nll
0,Model 1,1-30,[0:30),30,4,fixed_q=4,"dNch_deta, dET_deta",True,False,300,352,-1.698294e+03
1,Model 2,31-54,[30:54),24,4,fixed_q=4,"dN_dy_pion, dN_dy_kaon, dN_dy_proton",True,True,30,38,-3.549396e+03
2,Model 3,55-78,[54:78),24,4,fixed_q=4,"mean_pT_pion, mean_pT_kaon, mean_pT_proton",True,False,300,307,4.906661e+02
3,Model 4,79-98,[78:98),20,4,fixed_q=4,"pT_fluct, v22",True,True,241,264,1.226265e+07


In [28]:
results_table = metrics_df[
    [
        "model",
        "split",
        "column_range",
        "n_outputs",
        "q",
        "q_rule",
        "predrmse",
        "recrmse",
        "nrmse",
        "coverage",
        "width",
        "dss",
        "used_fast",
        "opt_success",
    ]
].sort_values(["model", "split"]).reset_index(drop=True)

results_table.round(4)


,model,split,column_range,n_outputs,q,q_rule,predrmse,recrmse,nrmse,coverage,width,dss,used_fast,opt_success
0,Model 1,test,1-30,30,4,fixed_q=4,95.8825,95.8825,0.0645,0.9113,461.8288,1041.2946,True,False
1,Model 1,train,1-30,30,4,fixed_q=4,39.0062,39.0062,0.0246,0.9890,265.5121,3239.1784,True,False
2,Model 2,test,31-54,24,4,fixed_q=4,37.9528,37.9528,0.0349,0.8677,108.1222,672.2125,True,True
3,Model 2,train,31-54,24,4,fixed_q=4,16.9341,16.9341,0.0152,0.9864,46.5641,1292.7335,True,True
4,Model 3,test,55-78,24,4,fixed_q=4,0.0314,0.0314,0.0371,0.6892,0.0555,-344.6930,True,False
5,Model 3,train,55-78,24,4,fixed_q=4,0.0066,0.0066,0.0075,0.8875,0.0186,-3984.2984,True,False
6,Model 4,test,79-98,20,4,fixed_q=4,0.0065,0.0065,0.0960,0.3927,0.0080,4249.9489,True,True
7,Model 4,train,79-98,20,4,fixed_q=4,0.0060,0.0060,0.0778,0.0590,0.0007,993176.1328,True,True


In [29]:
metric_cols = ["predrmse", "recrmse", "nrmse", "coverage", "width", "dss"]

metrics_df.pivot_table(
    index=["model", "column_range", "n_outputs", "q", "q_rule"],
    columns="split",
    values=metric_cols,
).round(4)


coverage                dss                nrmse         predrmse           recrmse              width          
split                                          test   train       test        train    test   train     test    train     test    train      test     train
model   column_range n_outputs q q_rule                                                                                                                    
Model 1 1-30         30        4 fixed_q=4   0.9113  0.9890  1041.2946    3239.1784  0.0645  0.0246  95.8825  39.0062  95.8825  39.0062  461.8288  265.5121
Model 2 31-54        24        4 fixed_q=4   0.8677  0.9864   672.2125    1292.7335  0.0349  0.0152  37.9528  16.9341  37.9528  16.9341  108.1222   46.5641
Model 3 55-78        24        4 fixed_q=4   0.6892  0.8875  -344.6930   -3984.2984  0.0371  0.0075   0.0314   0.0066   0.0314   0.0066    0.0555    0.0186
Model 4 79-98        20        4 fixed_q=4   0.3927  0.0590  4249.9489  993176.1328  0.0960  0.0778   0.0065   0.0060   0.0065   0.0060    0.0080    0.0007

In [10]:
run["assumption_note"]


'The nuclear dataset provides one observed output matrix only, so RRMSE is computed against all_f as a deterministic surrogate target.'